In [13]:
import pandas as pd
from pathlib import Path

NBA_df=pd.read_csv(Path.home()/"Desktop"/"NBA_Project"/"data"/"processed"/"nba_final_player_season_data.csv")
numeric_variables = NBA_df.select_dtypes(include="number").shape[1]

print(f"Original rows: {NBA_df.shape[0]:,}   Columns: {NBA_df.shape[1]}   Numeric variables: {numeric_variables}")

ANALYSIS_VARS=[
"age","career_start_year","player_experience_years","height_cm","weight_kg",
"draft_round","draft_number","college","country",
"team_wins","team_losses","team_win_pct","team_points_per_game",
"team_opp_points_per_game","team_point_diff_per_game",
"games_played","minutes","points","field_goal_attempts",
"two_point_attempts","three_point_attempts","free_throw_attempts",
"true_shooting_pct","rebounds","assists","turnovers","steals",
"blocks","personal_fouls","plus_minus","nba_fantasy_points",
"net_rating","usage_pct","assist_pct",
"offensive_rebound_pct","defensive_rebound_pct",
"actual_playoff_team"]

ANALYSIS_VARS=[x for x in ANALYSIS_VARS if x in NBA_df.columns]

missing=pd.DataFrame({
"n_missing":NBA_df[ANALYSIS_VARS].isna().sum(),
"share_pct":(NBA_df[ANALYSIS_VARS].isna().mean()*100).round(2)
}).query("n_missing>0").sort_values("n_missing",ascending=False)

print("="*60,"\nVARIABLES WITH MISSING VALUES\n","="*60,sep="")
print(missing.to_string() if len(missing) else "No missing values")

print("\n"+"="*60+"\nWITHIN-PLAYER MISSINGNESS\n"+"="*60)

for col in missing.index:
    m=NBA_df.groupby("player_id")[col].apply(lambda x:x.isna().mean())
    m=m[m>0]
    if len(m):
        always=(m==1).sum()
        partial=(m<1).sum()
        print(f"\n{col}:")
        print(" Missing in all seasons:",always)
        print(" Missing in some seasons:",partial)

if "college" in NBA_df.columns:
    print("\n"+"="*60+"\nCOLLEGE MISSINGNESS\n"+"="*60)
    players=NBA_df.drop_duplicates("player_id").copy()
    players["no_college"]=players["college"].isna()

    if "country" in players.columns:
        tab=pd.crosstab(
            players["country"].where(players["country"].eq("USA"),"non-USA"),
            players["no_college"],
            normalize="index"
        ).round(3)*100
        tab.columns=["has_college_pct","no_college_pct"]
        print(tab.to_string())

    if "career_start_year" in players.columns:
        era=pd.cut(players["career_start_year"],[0,1995,2005,2015,2100],labels=["pre-1996","1996-2005","2006-2015","2016+"])
        tab=pd.crosstab(era,players["no_college"],normalize="index").round(3)*100
        tab.columns=["has_college_pct","no_college_pct"]
        print("\nBy career start:")
        print(tab.to_string())


print("\n"+"="*60+"\nDRAFT MISSINGNESS\n"+"="*60)

if "draft_round" in NBA_df.columns:
    undrafted_label = NBA_df["draft_round"].astype(str).str.strip().eq("Undrafted").sum()
    undrafted_zero = pd.to_numeric(NBA_df["draft_round"], errors="coerce").eq(0).sum()

    print("\nUndrafted representation in draft_round:")
    print(f"  'Undrafted' label: {undrafted_label:,} observations")
    print(f"  Value of 0: {undrafted_zero:,} observations")

if "draft_number" in NBA_df.columns:
    undrafted_label = NBA_df["draft_number"].astype(str).str.strip().eq("Undrafted").sum()
    undrafted_zero = pd.to_numeric(NBA_df["draft_number"], errors="coerce").eq(0).sum()

    print("\nUndrafted representation in draft_number:")
    print(f"  'Undrafted' label: {undrafted_label:,} observations")
    print(f"  Value of 0: {undrafted_zero:,} observations")


print("\n"+"="*60+"\nGAMES PER SEASON\n"+"="*60)

season_col="season" if "season" in NBA_df.columns else "year"

schedule=NBA_df[["team_id",season_col,"team_wins","team_losses"]].drop_duplicates()
schedule["scheduled_games"]=schedule["team_wins"]+schedule["team_losses"]

season_table=schedule.groupby(season_col).agg(
    teams=("team_id","nunique"),
    min_games=("scheduled_games","min"),
    max_games=("scheduled_games","max"),
    median_games=("scheduled_games","median")
)

player_table=NBA_df.groupby(season_col).agg(
    players=("player_id","nunique"),
    median_player_games=("games_played","median")
)

season_table=season_table.join(player_table)

season_table["full_82_games"]=(season_table["min_games"]==82)&(season_table["max_games"]==82)

print(season_table.to_string())

print("\nNon-standard seasons:")
print(season_table[~season_table["full_82_games"]].to_string())

Original rows: 14,569   Columns: 67   Numeric variables: 56
VARIABLES WITH MISSING VALUES
              n_missing  share_pct
college            2038      13.99
draft_number        114       0.78
draft_round         102       0.70

WITHIN-PLAYER MISSINGNESS

college:
 Missing in all seasons: 369
 Missing in some seasons: 0

draft_number:
 Missing in all seasons: 59
 Missing in some seasons: 0

draft_round:
 Missing in all seasons: 57
 Missing in some seasons: 0

COLLEGE MISSINGNESS
         has_college_pct  no_college_pct
country                                 
USA                 97.1             2.9
non-USA             44.2            55.8

By career start:
                   has_college_pct  no_college_pct
career_start_year                                 
pre-1996                      97.5             2.5
1996-2005                     82.3            17.7
2006-2015                     85.8            14.2
2016+                         87.0            13.0

DRAFT MISSINGNESS

Undraf

In [12]:
season_col="season" if "season" in NBA_df.columns else "year"

team=NBA_df.groupby([season_col,"team"]).agg(
roster_minutes=("minutes","sum"),
roster_games=("games_played","sum"),
n_players=("player_id","nunique"),
wins=("team_wins","max"),
losses=("team_losses","max")
).reset_index()

team["scheduled"]=team["wins"]+team["losses"]
team["expected_minutes"]=team["scheduled"]*240
team["minutes_gap"]=team["roster_minutes"]-team["expected_minutes"]
team["gap_pct"]=(team["minutes_gap"]/team["expected_minutes"]*100).round(1)

print("Roster minutes imbalance:")
print(team["gap_pct"].describe().round(2))

print("\nMost over-credited teams:")
display(team.nlargest(10,"gap_pct")[[season_col,"team","gap_pct","n_players"]])

print("\nMost under-credited teams:")
display(team.nsmallest(10,"gap_pct")[[season_col,"team","gap_pct","n_players"]])

print("\nScale of imbalance:")
for t in [2,5,10,20]:
    n=(team["gap_pct"].abs()>t).sum()
    print(f">{t}%: {n} ({n/len(team)*100:.1f}%)")

print("\nRoster size:")
print(team["n_players"].describe().round(1))
print("Teams with >20 players:",(team["n_players"]>20).sum())

d=NBA_df.assign(sched=NBA_df["team_wins"]+NBA_df["team_losses"])

over=d[d["games_played"]>d["sched"]]

print("\nPlayers exceeding team schedule:",len(over))

if len(over):
    print(over.groupby(season_col).size())

Roster minutes imbalance:
count    892.00
mean       0.71
std        4.92
min      -29.00
25%       -1.00
50%        0.80
75%        3.02
max       17.40
Name: gap_pct, dtype: float64

Most over-credited teams:


,season,team,gap_pct,n_players
784,2022-23,LAC,17.4,19
542,2014-15,HOU,15.8,17
621,2016-17,WAS,14.9,17
229,2003-04,TOR,14.5,16
577,2015-16,MIA,14.5,15
862,2025-26,ATL,13.0,21
425,2010-11,MEM,12.9,16
537,2014-15,CLE,12.8,17
791,2022-23,NYK,12.8,17
513,2013-14,IND,12.3,15



Most under-credited teams:


,season,team,gap_pct,n_players
691,2019-20,GSW,-29.0,16
203,2003-04,ATL,-22.2,16
524,2013-14,PHI,-20.6,18
800,2022-23,UTA,-17.1,20
699,2019-20,MIN,-16.4,17
775,2022-23,CHA,-14.7,17
622,2017-18,ATL,-14.5,19
667,2018-19,MIA,-14.5,15
148,2001-02,CHI,-14.2,13
830,2023-24,UTA,-14.2,18



Scale of imbalance:
>2%: 463 (51.9%)
>5%: 233 (26.1%)
>10%: 50 (5.6%)
>20%: 3 (0.3%)

Roster size:
count    892.0
mean      16.3
std        2.2
min       12.0
25%       15.0
50%       16.0
75%       18.0
max       26.0
Name: n_players, dtype: float64
Teams with >20 players: 45

Players exceeding team schedule: 16
season
1996-97    3
1999-00    1
2000-01    1
2001-02    2
2002-03    1
2003-04    2
2004-05    1
2010-11    1
2013-14    1
2014-15    1
2022-23    1
2023-24    1
dtype: int64
